# Transformando labels

Top 10 SemEval:

label  
Loaded_Language                              1062  
Name_Calling-Labeling                         417  
Doubt                                         309  
Repetition                                    263  
Loaded_Language,Name_Calling-Labeling         218  
Appeal_to_Fear-Prejudice                      172  
Flag_Waving                                   139  
Exaggeration-Minimisation                     131  
Exaggeration-Minimisation,Loaded_Language      99  
Loaded_Language,Repetition                     90  

Vamos selecionar:  
1. Loaded_Language
2. Name_Calling-Labeling
3. Doubt
4. Repetition
5. Appeal_to_Fear-Prejudice
6. Flag_Waving
7. Exaggeration-Minimisation

**Referências:**  

1. Loaded language. Using specific words and phrases with strong emotional implications (either
positive or negative) to influence an audience (Weston, 2000, p. 6).  
2. Name calling or labeling. Labeling the object of the propaganda campaign as either something the
target audience fears, hates, finds undesirable or loves, praises (Miller, 1939).  
3. Doubt. Questioning the credibility of someone or something.  
4. Repetition. Repeating the same message over and over again, so that the audience will eventually
accept it (Torok, 2015; Miller, 1939).  
5. Appeal to fear/prejudice. Seeking to build support for an idea by instilling anxiety and/or panic in
the population towards an alternative, possibly based on preconceived judgments.  
6. Flag-waving. Playing on strong national feeling (or with respect to any group, e.g., race, gender,
political preference) to justify or promote an action or idea (Hobbs and Mcgee, 2008).  
7. Exaggeration or minimization. Either representing something in an excessive manner: making
things larger, better, worse or making something seem less important or smaller than it actually is (Jowett
and O’Donnell, 2012b, pag. 303).  


**Fonte:** https://aclanthology.org/2020.semeval-1.186.pdf

| Index | Label                      | Exemplo                                                                                                      |
|------:|--------------------------------|--------------------------------------------------------------------------------------------------------------|
| 1     | Loaded language               | Outrage as Donald Trump suggests injecting disinfectant to kill virus.                                      |
| 2     | Name calling, labeling        | WHO: Coronavirus emergency is ’Public Enemy Number 1’                                                         |
| 3     | Doubt                         | Can the same be said for the Obama Administration?                                                           |
| 4     | Repetition                    | I still have a dream. It is a dream deeply rooted in the American dream. I have a dream that one day ...     |
| 5     | Appeal to fear/prejudice      | A dark, impenetrable and “irreversible” winter of persecution of the faithful by their own shepherds will fall. |
| 6     | Flag-waving                   | Mueller attempts to stop the will of We the People!!! It’s time to jail Mueller.                             |
| 7     | Exaggeration, minimization    | Coronavirus ‘risk to the American people remains very low’, Trump said.                                      |
| 8     | No Label    | The meeting will take place at 3 PM tomorrow.                                      |

## Conexão Política

1. Selecionar frases com 3+ palavras
2. Tratar e normalizar textos
3. Fazer Labels com gpt

In [37]:
import pandas as pd
# make the text view larger
pd.set_option('display.max_colwidth', 100)


df = pd.read_csv("data/in/conexao_politica_sentences.csv", sep='|')
df = df.rename(columns={'sentence': 'text'})
df = df.drop_duplicates(subset=['text'])
df = df.reset_index(drop=True)

print(df.shape)
df.head(20)

(15866, 2)


,id,text
0,CNXP-opiniao-20250712-a1983d80-0,O presidente Luiz Inácio Lula da Silva (PT) minimizou nesta sexta-feira (11) a responsabilidade ...
1,CNXP-opiniao-20250712-a1983d80-1,"Em nova fala sobre o assunto, ele criticou a diferença entre o valor praticado pela Petrobras e ..."
2,CNXP-opiniao-20250712-a1983d80-2,"Durante cerimônia no Espírito Santo, Lula afirmou que ainda não recebeu uma justificativa convin..."
3,CNXP-opiniao-20250712-a1983d80-3,“Eu não me conformo que o gás de cozinha sai da Petrobras a R$ 37 e chega a R$ 140 nas casas dos...
4,CNXP-opiniao-20250712-a1983d80-4,"Alguém tem que me explicar”, disse o presidente em evento em Linhares (ES), durante o lançamento..."
5,CNXP-opiniao-20250712-a1983d80-5,"De acordo com dados da Petrobras, o preço médio do gás liquefeito de petróleo (GLP) nas refinari..."
6,CNXP-opiniao-20250712-a1983d80-6,"Já o preço médio do botijão ao consumidor gira em torno de R$ 108,31."
7,CNXP-opiniao-20250712-a1983d80-7,"A diferença, segundo especialistas, decorre de custos logísticos, margens de lucro das distribui..."
8,CNXP-opiniao-20250712-a1983d80-8,"Sem detalhar a fonte dos recursos, Lula voltou a fazer promessas em meio ao calendário eleitoral."
9,CNXP-opiniao-20250712-a1983d80-9,Reiterou que pretende ampliar o acesso gratuito ao gás de cozinha para mais de 17 milhões de pes...


In [38]:
# if we have less than 11 spaces, concat the text with its closest twin
# given we have sequential ids for same-source sentences, we can use the id to find the closest twin
# if its the first, we concat with the next one, if its the last or middle, we concat with the previous one

df['space_count'] = df['text'].str.count(' ')
print(df['space_count'].describe())

absorbed = set()   # rows that were merged INTO another row (drop them)
merged = set()     # rows that already merged someone (skip re-processing)

rows = df.set_index('id')['text'].to_dict()  # snapshot of original texts

for idx, row in df[df['space_count'] < 11].iterrows():
    row_id = row['id']
    if row_id in absorbed or row_id in merged:
        continue

    article, index = row_id.rsplit('-', 1)
    index = int(index)
    twin_id = f"{article}-{index + 1}" if index == 0 else f"{article}-{index - 1}"

    if twin_id in rows and twin_id not in absorbed:
        df.loc[idx, 'text'] = row['text'] + ' ' + rows[twin_id]
        absorbed.add(twin_id)
        merged.add(row_id)

df = df[~df['id'].isin(absorbed)].reset_index(drop=True)
df.head(20)

count    15866.000000
mean        20.950019
std         14.388709
min          0.000000
25%         11.000000
50%         18.000000
75%         27.000000
max        265.000000
Name: space_count, dtype: float64


,id,text,space_count
0,CNXP-opiniao-20250712-a1983d80-0,O presidente Luiz Inácio Lula da Silva (PT) minimizou nesta sexta-feira (11) a responsabilidade ...,27
1,CNXP-opiniao-20250712-a1983d80-1,"Em nova fala sobre o assunto, ele criticou a diferença entre o valor praticado pela Petrobras e ...",25
2,CNXP-opiniao-20250712-a1983d80-2,"Durante cerimônia no Espírito Santo, Lula afirmou que ainda não recebeu uma justificativa convin...",24
3,CNXP-opiniao-20250712-a1983d80-3,“Eu não me conformo que o gás de cozinha sai da Petrobras a R$ 37 e chega a R$ 140 nas casas dos...,23
4,CNXP-opiniao-20250712-a1983d80-4,"Alguém tem que me explicar”, disse o presidente em evento em Linhares (ES), durante o lançamento...",40
5,CNXP-opiniao-20250712-a1983d80-5,"De acordo com dados da Petrobras, o preço médio do gás liquefeito de petróleo (GLP) nas refinari...",28
6,CNXP-opiniao-20250712-a1983d80-6,"Já o preço médio do botijão ao consumidor gira em torno de R$ 108,31.",13
7,CNXP-opiniao-20250712-a1983d80-7,"A diferença, segundo especialistas, decorre de custos logísticos, margens de lucro das distribui...",20
8,CNXP-opiniao-20250712-a1983d80-8,"Sem detalhar a fonte dos recursos, Lula voltou a fazer promessas em meio ao calendário eleitoral.",15
9,CNXP-opiniao-20250712-a1983d80-9,Reiterou que pretende ampliar o acesso gratuito ao gás de cozinha para mais de 17 milhões de pes...,21


In [39]:
df[df['id'] == 'CNXP-opiniao-20250607-61b5a608-0']['text'].values[0]

'Hoje eu vou te ensinar como ser o novo Nikolas Ferreira. O segredo está aí, escancarado, mas você não viu porque estava ocupado demais se comparando com o engajamento alheio ou reclamando do algoritmo.'

In [40]:
df = df.drop(columns=['space_count'])
df['space_count'] = df['text'].str.count(' ')
print(df['space_count'].describe())


# order df by space count from biggest to smallest
df = df.sort_values(by='space_count', ascending=False).reset_index(drop=True)
df.head(20)

count    12295.000000
mean        25.832371
std         13.835127
min          2.000000
25%         16.000000
50%         23.000000
75%         31.000000
max        265.000000
Name: space_count, dtype: float64


,id,text,space_count
0,CNXP-opiniao-20190323-86c8d233-15,"E certamente com o risco de ser injusto com algumas pastas, faço questão de destacar os resultad...",265
1,CNXP-opiniao-20200224-bf494ff4-13,"Queda de 27,5% em relação ao mesmo período do ano passado, segundo os números Sinesp;\nCOMBATE À...",194
2,CNXP-opiniao-20180302-2842f74a-23,"A seguir, são sugeridas algumas medidas de combate às OCN e OCR, que não esgotam o rol das neces...",179
3,CNXP-opiniao-20180121-9bf38b8b-55,Cito onze aqui:\n– PL 6944/2017: impede a limitação dos dados de internet proposta pela Anatel e...,171
4,CNXP-opiniao-20180411-a78b3d07-10,"Um ex-presidente, dos mais populares e populistas, hoje responde pelo número 700004553820, numa ...",160
5,CNXP-opiniao-20210110-e57493fb-34,"2. que se a concepção e eventual gestação decorre da relação sexual sem o uso de contraceptivos,...",145
6,CNXP-opiniao-20180828-5a69d338-46,"De julho de 2018 quando esse artigo original foi escrito, até agora, quando ele foi atualizado, ...",131
7,CNXP-opiniao-20180831-a7048d75-120,[11] Gigantes de tecnologia negam censura de conteúdo político em redes sociais\n[12] Em audiênc...,129
8,CNXP-opiniao-20200914-df101eb3-3,"Ora, se de modo honesto e digno obtenho meios para prover a subsistência própria e de familiares...",129
9,CNXP-opiniao-20200331-d82e7f99-74,Essa transição está ocorrendo no meio de violentas lutas de classes e com elementos do capitalis...,129


In [41]:
df = df.drop(columns=['space_count'])
print(df.shape)
df.head(20)

(12295, 2)


,id,text
0,CNXP-opiniao-20190323-86c8d233-15,"E certamente com o risco de ser injusto com algumas pastas, faço questão de destacar os resultad..."
1,CNXP-opiniao-20200224-bf494ff4-13,"Queda de 27,5% em relação ao mesmo período do ano passado, segundo os números Sinesp;\nCOMBATE À..."
2,CNXP-opiniao-20180302-2842f74a-23,"A seguir, são sugeridas algumas medidas de combate às OCN e OCR, que não esgotam o rol das neces..."
3,CNXP-opiniao-20180121-9bf38b8b-55,Cito onze aqui:\n– PL 6944/2017: impede a limitação dos dados de internet proposta pela Anatel e...
4,CNXP-opiniao-20180411-a78b3d07-10,"Um ex-presidente, dos mais populares e populistas, hoje responde pelo número 700004553820, numa ..."
5,CNXP-opiniao-20210110-e57493fb-34,"2. que se a concepção e eventual gestação decorre da relação sexual sem o uso de contraceptivos,..."
6,CNXP-opiniao-20180828-5a69d338-46,"De julho de 2018 quando esse artigo original foi escrito, até agora, quando ele foi atualizado, ..."
7,CNXP-opiniao-20180831-a7048d75-120,[11] Gigantes de tecnologia negam censura de conteúdo político em redes sociais\n[12] Em audiênc...
8,CNXP-opiniao-20200914-df101eb3-3,"Ora, se de modo honesto e digno obtenho meios para prover a subsistência própria e de familiares..."
9,CNXP-opiniao-20200331-d82e7f99-74,Essa transição está ocorrendo no meio de violentas lutas de classes e com elementos do capitalis...


In [42]:
save = False

if save:
    df.to_csv("data/result/conexao-treated-sentences.csv", sep='|', index=False)

### Test Sample

In [43]:
# separate 200 sample to test it out
sample_df = df.sample(200, random_state=42).reset_index(drop=True)
print(sample_df.shape)
sample_df.head(5)

(200, 2)


,id,text
0,CNXP-opiniao-20190410-8d15f487-30,Uma empresa de Israel foi a primeira a desenvolver e instalar uma fábrica que trabalha só com en...
1,CNXP-opiniao-20200415-50930aa9-54,Isso trouxe a paz de volta à minha casa”. Ruth agradece a todos os parceiros que contribuem para...
2,CNXP-opiniao-20200614-e505f158-6,https://twitter.com/brenolaerte/status/1269776928245981190\nEsses atos de destruição de moviment...
3,CNXP-opiniao-20190805-103b47da-34,"""O pensamento de um vizinho desconhecido é válido por si só. Isso não exclui que alguém a tenha ..."
4,CNXP-opiniao-20190907-d494833e-52,"Você vive essa campanha? E os amigos próximos, numa conversa olho no olho?"


#### Prompt 1

In [44]:
from agno_labeler import create_labeler, process_label_batch

agent = create_labeler(
    model_name="gpt-4.1-nano",
    temperature=0.1,
    max_tokens=150,
    prompt_path="prompt/semeval-label-prompt.txt"
)

In [45]:
run = False

if run:
    await process_label_batch(
        sentences=sample_df['text'].tolist(),
        agent=agent,
        output_file="data/result/conexao-labels-test.csv",
        batch_size=5
    )

In [46]:
results = pd.read_csv("data/result/conexao-labels-test.csv")
print(results.shape)
results.head(5)

(200, 3)


,sentence,label,justification
0,Uma empresa de Israel foi a primeira a desenvolver e instalar uma fábrica que trabalha só com en...,No_Label,"The sentence presents factual information without emotional language, labels, or persuasive tech..."
1,Isso trouxe a paz de volta à minha casa”. Ruth agradece a todos os parceiros que contribuem para...,No_Label,"The sentence expresses gratitude and describes a positive outcome, but does not employ any speci..."
2,https://twitter.com/brenolaerte/status/1269776928245981190\nEsses atos de destruição de moviment...,Loaded_Language,"The sentence uses emotionally charged words like 'destruição', 'caos', and 'destruição' to evoke..."
3,"""O pensamento de um vizinho desconhecido é válido por si só. Isso não exclui que alguém a tenha ...",No_Label,The sentence presents a neutral statement about a neighbor’s thoughts and observations without e...
4,"Você vive essa campanha? E os amigos próximos, numa conversa olho no olho?",No_Label,The sentence questions personal engagement and social interactions without employing any specifi...


In [47]:
results['label'].value_counts()

label
Loaded_Language              75
No_Label                     61
Exaggeration-Minimisation    27
Flag_Waving                  15
Repetition                    9
Doubt                         8
Appeal_to_Fear-Prejudice      4
Name_Calling-Labeling         1
Name: count, dtype: int64

#### Prompt 2

In [3]:
from agno_labeler import create_labeler, process_label_batch

agent = create_labeler(
    model_name="gpt-4.1-nano",
    temperature=0.1,
    max_tokens=150,
    prompt_path="prompt/semeval-label-prompt2.txt"
)

In [49]:
run = False

if run:
    await process_label_batch(
        sentences=sample_df['text'].tolist(),
        agent=agent,
        output_file="data/result/conexao-labels-test2.csv",
        batch_size=10
    )

In [50]:
results = pd.read_csv("data/result/conexao-labels-test2.csv")
print(results.shape)
results.head(5)

(200, 3)


,sentence,label,justification
0,Uma empresa de Israel foi a primeira a desenvolver e instalar uma fábrica que trabalha só com en...,No_Label,"The sentence is purely informational without emotional, persuasive, or evaluative language."
1,Isso trouxe a paz de volta à minha casa”. Ruth agradece a todos os parceiros que contribuem para...,No_Label,The sentence is a personal expression of gratitude and does not contain persuasive language or t...
2,https://twitter.com/brenolaerte/status/1269776928245981190\nEsses atos de destruição de moviment...,Loaded_Language,The phrase 'criar o caos e trazer a destruição' uses strong emotional words to influence percept...
3,"""O pensamento de um vizinho desconhecido é válido por si só. Isso não exclui que alguém a tenha ...",No_Label,The sentence presents a neutral statement about a neighbor’s thoughts and observations without e...
4,"Você vive essa campanha? E os amigos próximos, numa conversa olho no olho?",No_Label,"The sentence is a rhetorical question about personal experience and social interactions, with no..."


In [51]:
results['label'].value_counts()

label
No_Label                     82
Loaded_Language              67
Exaggeration-Minimisation    17
Flag_Waving                  11
Doubt                        10
Name_Calling-Labeling         6
Appeal_to_Fear-Prejudice      5
Repetition                    2
Name: count, dtype: int64

#### Compare both results

In [52]:
df_compare = pd.merge(
    pd.read_csv("data/result/conexao-labels-test.csv"),
    pd.read_csv("data/result/conexao-labels-test2.csv"),
    on='sentence',
)
print(df_compare.shape)
df_compare.head(5)

(200, 5)


,sentence,label_x,justification_x,label_y,justification_y
0,Uma empresa de Israel foi a primeira a desenvolver e instalar uma fábrica que trabalha só com en...,No_Label,"The sentence presents factual information without emotional language, labels, or persuasive tech...",No_Label,"The sentence is purely informational without emotional, persuasive, or evaluative language."
1,Isso trouxe a paz de volta à minha casa”. Ruth agradece a todos os parceiros que contribuem para...,No_Label,"The sentence expresses gratitude and describes a positive outcome, but does not employ any speci...",No_Label,The sentence is a personal expression of gratitude and does not contain persuasive language or t...
2,https://twitter.com/brenolaerte/status/1269776928245981190\nEsses atos de destruição de moviment...,Loaded_Language,"The sentence uses emotionally charged words like 'destruição', 'caos', and 'destruição' to evoke...",Loaded_Language,The phrase 'criar o caos e trazer a destruição' uses strong emotional words to influence percept...
3,"""O pensamento de um vizinho desconhecido é válido por si só. Isso não exclui que alguém a tenha ...",No_Label,The sentence presents a neutral statement about a neighbor’s thoughts and observations without e...,No_Label,The sentence presents a neutral statement about a neighbor’s thoughts and observations without e...
4,"Você vive essa campanha? E os amigos próximos, numa conversa olho no olho?",No_Label,The sentence questions personal engagement and social interactions without employing any specifi...,No_Label,"The sentence is a rhetorical question about personal experience and social interactions, with no..."


In [53]:
# see where the labels differ
pd.set_option('display.max_colwidth', 200)
print(df_compare[df_compare['label_x'] != df_compare['label_y']].shape)
df_compare[df_compare['label_x'] != df_compare['label_y']].head(10)

(48, 5)


,sentence,label_x,justification_x,label_y,justification_y
6,"Em 15 de janeiro de 2018, foi morto extrajudicialmente durante uma operação das forças de segurança do estado que cercou a casa onde ele e outras pessoas estavam.",Exaggeration-Minimisation,"The phrase 'foi morto extrajudicialmente durante uma operação' emphasizes the extrajudicial killing, which could be seen as an exaggeration of the event's nature to evoke a strong emotional response.",No_Label,The sentence is a factual statement describing an event without emotional language or persuasion techniques.
7,"A direção que a bíblia propõe é que a mergulhemos em nosso mundo interior, para que por meio das Suas palavras de vida eterna descubramos duas coisas: quem somos e como resgatar nossa identidade o...",Loaded_Language,"The sentence uses emotionally charged words like 'propõe', 'mergulhemos', 'palavras de vida eterna', 'descubramos', 'quem somos', and 'resgatar nossa identidade original', which evoke a sense of s...",No_Label,"The sentence is informational and spiritual in tone, without emotional language, labels, doubt, repetition, fear appeal, flag-waving, or exaggeration."
10,"De fato, quatro foram as forças sociais que alçaram o Capitão à Presidência da República para instalar no Brasil um governo liberal na economia e conservador nos costumes, a saber: movimento conse...",Loaded_Language,"The phrase 'alçaram o Capitão à Presidência' uses strong, elevating language ('alçaram') to emphasize support, appealing to positive emotional connotations.",Repetition,"The phrase 'a saber' introduces a list, emphasizing the four forces, which is a form of repetition to reinforce the points."
13,"Tudo isso longe do cabresto estatal. E muitos desses atalhos passam pela lógica privada, pela tecnologia, pela autonomia individual.",No_Label,"The sentence discusses concepts like state control, private logic, technology, and individual autonomy without employing any specific propaganda technique.",Flag_Waving,"The sentence emphasizes 'autonomia individual' and 'longo do cabresto estatal,' appealing to national or group pride in independence and freedom from state control."
16,"O pior tipo de crime que se pode imaginar. Que ninguém jamais esqueça: PT, PCB, PCdoB, PSOL, REDE e outros partidos e líderes de esquerda, especialmente Lula, apoiaram explicitamente o regime de N...",Loaded_Language,The sentence uses emotionally charged words like 'O pior tipo de crime' and 'crimes contra a humanidade' to evoke strong emotional reactions.,Name_Calling-Labeling,"The sentence labels leftist parties and leaders as supporting a regime that commits crimes against humanity, which is an attack on their character and credibility."
20,"O primeiro-ministro israelense, Benjamin Netanyahu, assumiu a campanha e usou seu excelente relacionamento com Putin para promover a libertação, até agora sem resultado.",Repetition,"The sentence emphasizes Netanyahu's actions and relationships, repeatedly highlighting his campaign and connections to reinforce his efforts.",No_Label,"The sentence is informational, describing actions and relationships without employing emotional language, labels, or persuasive techniques."
24,"Será que é preciso morrer um ente querido seu por falta de respirador, pra você entender que quem rouba o dinheiro da saúde roubou o oxigênio daquele que você amava?",Doubt,"The sentence questions the credibility of those stealing health funds by implying it results in deaths, using the phrase 'Será que é preciso morrer um ente querido seu' to cast doubt on the cause-...",Appeal_to_Fear-Prejudice,"The sentence evokes fear and emotional pain by suggesting that theft in health funds leads to death, aiming to influence perception through emotional impact."
27,"Daí aparece esse tal de Marco Antônio Villa e diz que, quando chegar ao poder, Bolsonaro vai criar o Petrolão 2.",Exaggeration-Minimisation,"The sentence suggests that Bolsonaro will create a new scandal ('Petrolão 2'), which is an exaggerated 

### Label Whole dataset

In [54]:
new_df = df.copy()
print(new_df.shape)

new_df.head(5)

(12295, 2)


,id,text
0,CNXP-opiniao-20190323-86c8d233-15,"E certamente com o risco de ser injusto com algumas pastas, faço questão de destacar os resultados positivos já apresentados:\nFormação de uma equipe de Notáveis, com Ministros escolhidos pela qua..."
1,CNXP-opiniao-20200224-bf494ff4-13,"Queda de 27,5% em relação ao mesmo período do ano passado, segundo os números Sinesp;\nCOMBATE ÀS DROGAS: Aumento de 158% na apreensão de cocaína no primeiro semestre deste ano em comparação ao me..."
2,CNXP-opiniao-20180302-2842f74a-23,"A seguir, são sugeridas algumas medidas de combate às OCN e OCR, que não esgotam o rol das necessárias: \n- Endurecer a lei sobre Organização Criminosa, tornando a justiça ágil e mais rigorosa;\n-..."
3,CNXP-opiniao-20180121-9bf38b8b-55,Cito onze aqui:\n– PL 6944/2017: impede a limitação dos dados de internet proposta pela Anatel e as empresas concessionárias que prestam serviços de banda larga\n– PL-4730/2016: torna hediondos os...
4,CNXP-opiniao-20180411-a78b3d07-10,"Um ex-presidente, dos mais populares e populistas, hoje responde pelo número 700004553820, numa cela improvisada na sede da Polícia Federal em Curitiba, intencionando ser transferido para um quart..."


In [19]:
import pandas as pd

new_df = pd.read_csv("data/result/conexao-treated-sentences.csv", sep='|')
new_df2 = pd.read_csv("data/result/conexao-labels-full.csv")

# get the texts that already were processed
processed_texts = set(new_df2['sentence'].tolist())
to_process = new_df[~new_df['text'].isin(processed_texts)]

# add ERROR labels from new_df into to_process, and save the new_df again
to_process = pd.concat([to_process, new_df2[new_df2['label'] == 'ERROR']], ignore_index=True)

resave = True
if resave:
    new_df2[new_df2['label'] != 'ERROR'].to_csv("data/result/conexao-labels-full.csv", sep='|', index=False)

print(to_process.shape)
to_process.head(5)

(48, 5)


,id,text,sentence,label,justification
0,NaN,NaN,Mas essa é uma outra lição dada pelos revoluci...,ERROR,"Failed to parse: {""label"":""Name_Calling-Labeli..."
1,NaN,NaN,Exemplo: lei da ficha limpa proíbe candidato d...,ERROR,Rate limit reached for gpt-4.1-nano in organiz...
2,NaN,NaN,"A resposta do autor do estudo, Mandeep Mehra, ...",ERROR,Rate limit reached for gpt-4.1-nano in organiz...
3,NaN,NaN,Mas tudo indica que não foi. 3) Ser um empreen...,ERROR,Rate limit reached for gpt-4.1-nano in organiz...
4,NaN,NaN,"Observa, sem entender muito se está assistindo...",ERROR,Rate limit reached for gpt-4.1-nano in organiz...


In [20]:
run = True

if run:
    await process_label_batch(
        sentences=to_process['text'].tolist(),
        agent=agent,
        output_file="data/result/conexao-labels-full.csv",
        batch_size=10
    )

Processing batch 5: 100%|██████████| 8/8 [00:15<00:00,  1.94s/it]


In [22]:
df = pd.read_csv("data/result/conexao-labels-full.csv", sep='|')
print(df.shape)
df.head(5)

(12335, 3)


,sentence,label,justification
0,E certamente com o risco de ser injusto com al...,Loaded_Language,The sentence uses emotionally charged words li...
1,"Queda de 27,5% em relação ao mesmo período do ...",No_Label,The sentences are factual reports and do not c...
2,"A seguir, são sugeridas algumas medidas de com...",No_Label,The sentence is informational and describes me...
3,Cito onze aqui:\n– PL 6944/2017: impede a limi...,No_Label,"The sentence is informational, listing legisla..."
4,"Um ex-presidente, dos mais populares e populis...",Loaded_Language,The sentence uses emotionally charged words li...


In [23]:
df['label'].value_counts()

label
No_Label                     5060
Loaded_Language              4252
Exaggeration-Minimisation     761
Doubt                         758
Flag_Waving                   696
Name_Calling-Labeling         366
Repetition                    234
Appeal_to_Fear-Prejudice      160
Name: count, dtype: int64

## G1

1. Tratar e selecionar textos com base em algum padrão
2. Fazer Labels com gpt

In [17]:
import pandas as pd

df = pd.read_csv("data/in/g1_extracted_blockquotes_reasonings.csv", sep='|')
df = df.rename(columns={'blockquote_text': 'text', 'next_item_text': 'justification'})[['text', 'status']]
df = df.drop_duplicates(subset=['text'])
df = df.reset_index(drop=True)
print(df.shape)
df.head()

(3668, 2)


,text,status
0,"O vídeo foi gravado no cemitério de Annotto Bay, na região nordeste do país. Segundo uma reporta...",FATO
1,OFato ou Fakepesquisou pelo cemitério de Annotto Bay no Google Maps e confirmou que os totens sã...,FATO
2,"Mas isso não é verdade.OFato ou Fakeenviou o material à assessoria de imprensa do TSE, que respo...",FAKE
3,"Segundo o ex-ministro do TSE Carlos Horbach, relator do caso em 2022,a consulta da AGU foi rejei...",FAKE
4,"Não há, no texto do ECA Digital, qualquer menção ao bloqueio de aparelhos eletrônicos ou de sist...",FAKE


In [18]:
# get texts that start with '"'
starts_with_quote = df[df['text'].str.startswith('"')].copy().reset_index(drop=True)
print(starts_with_quote.shape)
starts_with_quote.tail(20)

(1158, 2)


,text,status
1138,"""Na pandemia, os empresários do setor [de transporte coletivo] fizeram o que quiseram, reduziram...",FAKE
1139,"""O ministro Tarcísio e o presidente Bolsonaro sinalizaram esta vontade de empenhar este recurso ...",FAKE
1140,"""Nossa floresta é úmida e não permite a propagação do fogo em seu interior. Os incêndios acontec...",NAOEBEMASSIM
1141,"""Ninguém pode obrigar ninguém a tomar vacina""",FAKE
1142,"""As fibras do tecido não são completamente vedadas, então há espaço e por ali passa ar que entra...",FAKE
1143,"""O Rio de Janeiro na véspera do Dia dos Pais era o único ponto azul [em queda] do mapa [de morte...",FATO
1144,"""Muitas dessas informações falsas podem não ajudar nem prejudicar, apenas causar alarde. Porém, ...",FATO
1145,"""É preciso compreender, antes de mais nada, como funcionam os medicamentos e como eles são pesqu...",FATO
1146,"""As máscaras são utilizadas há décadas pelos profissionais de saúde, seja nas cirurgias, seja em...",FATO
1147,"""Mensagens falsas sobre o uso de máscaras que tentam convencer pessoas a não usá-las representa ...",FATO


In [20]:
new_df = starts_with_quote.copy()
print(new_df.shape)
new_df.head(5)

(1158, 2)


,text,status
0,"""Esclarecemos que se trata de uma montagem, que não correspondem aos preços reais praticados em ...",FAKE
1,"""O que chamou atenção foi o comportamento do canguru: ele aparece deitado de barriga para cima, ...",FATO
2,"""Esse tipo de contato pode causar estresse, alterar comportamentos naturais e até representar ri...",FATO
3,"""Recentemente, obtivemos informações não verificadas de que, no início de fevereiro de 2026, o I...",FATO
4,"""Estamos cientes dessas informações. ... Trata-se de manter uma postura de preparação para cenár...",FATO


In [21]:
new_df['space_count'] = new_df['text'].str.count(' ')
print(new_df['space_count'].describe())

count    1158.000000
mean       34.599309
std        21.460423
min         3.000000
25%        18.000000
50%        30.000000
75%        47.000000
max       183.000000
Name: space_count, dtype: float64


In [22]:
save = True

if save:
    df.to_csv("data/result/g1-treated-texts.csv", sep='|', index=False)

In [23]:
from agno_labeler import create_labeler, process_label_batch

agent = create_labeler(
    model_name="gpt-4.1-nano",
    temperature=0.1,
    max_tokens=150,
    prompt_path="prompt/semeval-label-prompt.txt"
)

In [27]:
df = pd.read_csv("data/result/g1-labels-full.csv")
print(df.shape)
df.head(5)

(3871, 3)


,sentence,label,justification
0,O vídeo foi gravado no cemitério de Annotto Ba...,No_Label,"The sentence is purely informational, describi..."
1,OFato ou Fakepesquisou pelo cemitério de Annot...,No_Label,The sentence provides factual information abou...
2,Mas isso não é verdade.OFato ou Fakeenviou o m...,No_Label,The sentence primarily relays factual informat...
3,"Segundo o ex-ministro do TSE Carlos Horbach, r...",No_Label,The sentence is a detailed explanation of a le...
4,"Não há, no texto do ECA Digital, qualquer menç...",No_Label,The sentence provides an informational clarifi...


In [31]:
df = pd.read_csv("data/result/g1-treated-texts.csv", sep='|')
print(df.shape)
df.head(5)

(3668, 2)


,text,status
0,O vídeo foi gravado no cemitério de Annotto Ba...,FATO
1,OFato ou Fakepesquisou pelo cemitério de Annot...,FATO
2,Mas isso não é verdade.OFato ou Fakeenviou o m...,FAKE
3,"Segundo o ex-ministro do TSE Carlos Horbach, r...",FAKE
4,"Não há, no texto do ECA Digital, qualquer menç...",FAKE


In [29]:
df['label'].value_counts()

label
No_Label                     1660
Exaggeration-Minimisation     791
Loaded_Language               785
ERROR                         203
Flag_Waving                   159
Doubt                         116
Repetition                    102
Name_Calling-Labeling          38
Appeal_to_Fear-Prejudice       17
Name: count, dtype: int64

## SemEval

1. Selecionar frases com 3+ palavras
2. Traduzir textos pro português (?) OPCIONAL
3. Selecionar apenas 1 label para cada texto

In [ ]:
import pandas as pd

df = pd.read_csv("data/in/task3-df.csv")
print(df.shape)
df.head()

(12291, 4)


,article_id,text_id,label,text
0,813452859,1,NaN,EU Profits From Trading With UK While London L...
1,813452859,3,NaN,With the Parliamentary vote on British Prime M...
2,813452859,4,NaN,But is there any chance that May's deal will m...
3,813452859,5,NaN,Sputnik spoke with political campaigner Michae...
4,813452859,6,NaN,Sputnik: Does Theresa May have any chance of g...


In [51]:
df['label'].value_counts().head(10)

label
Loaded_Language                              1062
Name_Calling-Labeling                         417
Doubt                                         309
Repetition                                    263
Loaded_Language,Name_Calling-Labeling         218
Appeal_to_Fear-Prejudice                      172
Flag_Waving                                   139
Exaggeration-Minimisation                     131
Exaggeration-Minimisation,Loaded_Language      99
Loaded_Language,Repetition                     90
Name: count, dtype: int64

In [52]:
selected_labels = [
    "Loaded_Language",
    "Name_Calling-Labeling",
    "Doubt",
    "Repetition",
    "Appeal_to_Fear-Prejudice",
    "Flag_Waving",
    "Exaggeration-Minimisation"
]

In [53]:
# separate the datasets into -> contains_comma, doesnot_contains_comma
# if ',' in df label
contains_comma = df[df['label'].str.contains(',', na=False)].copy()
doesnot_contains_comma = df[df['label'].str.contains(',', na=False) == False].copy()
has_na = df[df['label'].isna()].copy()

In [54]:
print(doesnot_contains_comma.shape)
doesnot_contains_comma['label'].value_counts(dropna=False).head(50)

(10532, 4)


label
NaN                                7578
Loaded_Language                    1062
Name_Calling-Labeling               417
Doubt                               309
Repetition                          263
Appeal_to_Fear-Prejudice            172
Flag_Waving                         139
Exaggeration-Minimisation           131
Appeal_to_Authority                  77
False_Dilemma-No_Choice              77
Causal_Oversimplification            69
Slogans                              68
Conversation_Killer                  52
Red_Herring                          27
Appeal_to_Popularity                 22
Guilt_by_Association                 19
Appeal_to_Hypocrisy                  18
Obfuscation-Vagueness-Confusion      16
Straw_Man                            12
Whataboutism                          4
Name: count, dtype: int64

In [55]:
# keep only selected labels
doesnot_contains_comma = doesnot_contains_comma[doesnot_contains_comma['label'].isin(selected_labels)].copy()
print(doesnot_contains_comma.shape)
doesnot_contains_comma['label'].value_counts().head(10)

(2493, 4)


label
Loaded_Language              1062
Name_Calling-Labeling         417
Doubt                         309
Repetition                    263
Appeal_to_Fear-Prejudice      172
Flag_Waving                   139
Exaggeration-Minimisation     131
Name: count, dtype: int64

In [56]:
print(contains_comma.shape)
contains_comma['label'].value_counts().head(10)

(1759, 4)


label
Loaded_Language,Name_Calling-Labeling                              218
Exaggeration-Minimisation,Loaded_Language                           99
Loaded_Language,Repetition                                          90
Doubt,Loaded_Language                                               89
Doubt,Name_Calling-Labeling                                         52
Appeal_to_Fear-Prejudice,Loaded_Language                            44
Name_Calling-Labeling,Repetition                                    43
Exaggeration-Minimisation,Loaded_Language,Name_Calling-Labeling     41
Exaggeration-Minimisation,Name_Calling-Labeling                     37
Flag_Waving,Loaded_Language                                         36
Name: count, dtype: int64

In [57]:
# explode commas into new label columns
# eg.: Loaded_Language,Name_Calling-Labeling -> Loaded_Language, Name_Calling-Labeling
# so the dataset will have 1-N new columns depending on the max number of commas a label has
# label0, label1, label2, ...
contains_comma['label'] = contains_comma['label'].str.split(',')
contains_comma = contains_comma.explode('label')
print(contains_comma.shape)
display(contains_comma.head(5))
contains_comma['label'].value_counts().head(10)

(4291, 4)


,article_id,text_id,label,text
5,813452859,7,False_Dilemma-No_Choice,Michael Swadling: I guess her only chance is i...
5,813452859,7,Loaded_Language,Michael Swadling: I guess her only chance is i...
7,813452859,9,False_Dilemma-No_Choice,There is a chance; as unfortunately there are ...
7,813452859,9,Loaded_Language,There is a chance; as unfortunately there are ...
7,813452859,9,Name_Calling-Labeling,There is a chance; as unfortunately there are ...


label
Loaded_Language              1148
Name_Calling-Labeling         767
Exaggeration-Minimisation     423
Repetition                    394
Doubt                         370
Appeal_to_Fear-Prejudice      261
Flag_Waving                   237
Causal_Oversimplification     158
False_Dilemma-No_Choice       105
Slogans                       105
Name: count, dtype: int64

In [58]:
# keep only selected labels
contains_comma = contains_comma[contains_comma['label'].isin(selected_labels)].copy()
print(contains_comma.shape)
contains_comma['label'].value_counts().head(10)

(3600, 4)


label
Loaded_Language              1148
Name_Calling-Labeling         767
Exaggeration-Minimisation     423
Repetition                    394
Doubt                         370
Appeal_to_Fear-Prejudice      261
Flag_Waving                   237
Name: count, dtype: int64

In [59]:
print(has_na.shape)
has_na.head()

(7578, 4)


,article_id,text_id,label,text
0,813452859,1,NaN,EU Profits From Trading With UK While London L...
1,813452859,3,NaN,With the Parliamentary vote on British Prime M...
2,813452859,4,NaN,But is there any chance that May's deal will m...
3,813452859,5,NaN,Sputnik spoke with political campaigner Michae...
4,813452859,6,NaN,Sputnik: Does Theresa May have any chance of g...


In [60]:
# change all na labels to 'No_Label'
has_na['label'] = 'No_Label'
print(has_na.shape)
has_na['label'].value_counts().head(10)

(7578, 4)


label
No_Label    7578
Name: count, dtype: int64

### Concat Everything

In [61]:
new_df = pd.concat([doesnot_contains_comma, contains_comma, has_na], ignore_index=True)
print(new_df.shape)
new_df['label'].value_counts().head(10)

(13671, 4)


label
No_Label                     7578
Loaded_Language              2210
Name_Calling-Labeling        1184
Doubt                         679
Repetition                    657
Exaggeration-Minimisation     554
Appeal_to_Fear-Prejudice      433
Flag_Waving                   376
Name: count, dtype: int64

In [62]:
# difference between No_Label and sum of the other labels
print(new_df[new_df['label'] == 'No_Label'].shape[0])
print(new_df[new_df['label'] != 'No_Label'].shape[0])

7578
6093


In [106]:
_ = new_df.copy()

_['space_count'] = _['text'].str.count(' ')
print(_.groupby('label')['space_count'].describe())
print()
print(_['space_count'].describe())

                            count       mean         std  min   25%   50%  \
label                                                                       
Appeal_to_Fear-Prejudice    433.0  58.420323   76.906020  4.0  23.0  40.0   
Doubt                       679.0  52.985272   60.646685  0.0  21.5  38.0   
Exaggeration-Minimisation   554.0  65.774368   73.639349  0.0  27.0  46.0   
Flag_Waving                 376.0  51.534574   48.218601  3.0  22.0  41.0   
Loaded_Language            2210.0  61.208597  202.347892  0.0  25.0  44.0   
Name_Calling-Labeling      1184.0  57.463682   57.738139  2.0  25.0  44.0   
No_Label                   7578.0  28.007126   36.295230  0.0  10.0  21.0   
Repetition                  657.0  59.496195   70.842020  0.0  23.0  41.0   

                            75%     max  
label                                    
Appeal_to_Fear-Prejudice   73.0  1030.0  
Doubt                      60.0   627.0  
Exaggeration-Minimisation  75.0   776.0  
Flag_Waving        

In [63]:
save = True

if save:
    new_df.to_csv("data/out/semeval-treated.csv", index=False)